In [ ]:
import csv
from heapq import nsmallest
from typing import List, Tuple, Optional
import pandas as pd


def _sniff_delimiter(path: str, sample_bytes: int = 1_000_000) -> str:
    with open(path, 'rb') as fb:
        sample = fb.read(sample_bytes)
    try:
        dialect = csv.Sniffer().sniff(sample.decode('utf-8', errors='ignore'))
        return dialect.delimiter
    except Exception:
        text = sample.decode('utf-8', errors='ignore')
        candidates = [',', ';', '\t', '|']
        delim = max(candidates, key=lambda d: text.count(d))
        return delim


def _normalize_header(names: List[str]) -> List[str]:
    norm = []
    for n in names:
        if n is None:
            norm.append('')
        else:
            n2 = n.strip().strip('\ufeff').lower().replace(' ', '_')
            norm.append(n2)
    return norm


def _resolve_columns(fieldnames: List[str]) -> Optional[Tuple[str, str, str, str, str]]:
    fn = set(fieldnames)
    def pick(cands):
        for c in cands:
            if c in fn:
                return c
        return None

    col_i = pick(['i', 'src', 'source', 'game_i', 'from'])
    col_j = pick(['j', 'dst', 'target', 'game_j', 'to'])
    col_shared = pick(['shared_users', 'shared', 'co_count', 'n_common'])
    col_cos = pick(['cosine', 'cos', 'cosine_sim', 'cos_sim'])
    col_jac = pick(['jaccard', 'jac', 'jaccard_sim', 'jac_sim'])

    if col_i and col_j and col_cos and col_jac:
        if not col_shared:
            col_shared = ''
        return col_i, col_j, col_shared, col_cos, col_jac
    return None


# Cached lookups for matrix index ↔ bgg_id and bgg_id → name
_UNIQUE_ITEMS_CACHE = None
_ID_TO_NAME_CACHE = None


def _get_unique_items():
    """Get or reconstruct unique_items mapping (matrix_index → bgg_id)"""
    global _UNIQUE_ITEMS_CACHE
    
    # First, try to use global unique_items if available
    if 'unique_items' in globals():
        return globals()['unique_items']
    
    # If cached, return it
    if _UNIQUE_ITEMS_CACHE is not None:
        return _UNIQUE_ITEMS_CACHE
    
    # Otherwise, reconstruct it from the rating data
    try:
        df = pd.read_csv('../data/bgg_rating_threshold.csv', sep=';', usecols=['bgg_id'])
        _, unique_items = pd.factorize(df['bgg_id'], sort=True)
        _UNIQUE_ITEMS_CACHE = unique_items
        return unique_items
    except Exception as e:
        print(f"Warning: Could not reconstruct unique_items: {e}")
        return None


def _get_id_to_name():
    """Get name mapping (bgg_id → game_name)"""
    global _ID_TO_NAME_CACHE
    
    # First, try to use global id_to_name if available
    if 'id_to_name' in globals():
        return globals()['id_to_name']
    
    # If cached, return it
    if _ID_TO_NAME_CACHE is not None:
        return _ID_TO_NAME_CACHE
    
    # Otherwise, load from boardgames_ranks.csv
    try:
        df_game_item_from_bgg = pd.read_csv('../data/boardgames_ranks.csv')
        id_to_name = df_game_item_from_bgg.set_index('id')['name']
        _ID_TO_NAME_CACHE = id_to_name
        return id_to_name
    except Exception as e:
        print(f"Warning: Could not load id_to_name: {e}")
        return None


def topk_impact_from_csv(
    k: Optional[int] = None,
    max_print: int = 10,
):
    """Compute impact using the preloaded cache (no CSV re-read).

    - If `k` is None, uses the preload `k`.
    - If `k` is provided, it must be <= preload `k`.
    """
    if '_BGG_CACHE' not in globals() or _BGG_CACHE is None:
        raise RuntimeError(
            "Cache not found. Run `preload_streamed_index(...)` first to read the CSV once."
        )

    cache = _BGG_CACHE
    use_k = cache.k if k is None else k
    if use_k > cache.k:
        raise ValueError(
            f"Requested k={use_k} exceeds preloaded k={cache.k}. Re-run preload with larger k."
        )

    impacted = []
    # Note: heaps in cache contain at most cache.k entries; we take top `use_k`
    for i, heap_c in cache.topk_cosine.items():
        heap_j = cache.topk_jaccard.get(i, [])
        set_c = {j for _, j in nsmallest(use_k, heap_c)}
        set_j = {j for _, j in nsmallest(use_k, heap_j)}
        inter = len(set_c & set_j)
        disagree = len(set_c) + len(set_j) - 2 * inter

        sorted_c = sorted(heap_c, key=lambda x: x[0], reverse=True)[:use_k]
        rankC = {j: r for r, (_, j) in enumerate(sorted_c, 1)}
        sorted_j = sorted(heap_j, key=lambda x: x[0], reverse=True)[:use_k]
        rankJ = {j: r for r, (_, j) in enumerate(sorted_j, 1)}
        pos_diff_sum = sum(abs(rankC[j] - rankJ[j]) for j in rankC.keys() & rankJ.keys())

        impacted.append(
            (
                disagree,
                pos_diff_sum,
                i,
                inter,
                [p[1] for p in sorted_c],
                [p[1] for p in sorted_j],
            )
        )

    print(
        f"Computing impact for {len(cache.topk_cosine):,} sources with k={use_k} (preloaded k={cache.k})…",
        flush=True,
    )
    impacted.sort(reverse=True)

    best_i = impacted[0][2] if impacted else None

    # Get mappings
    unique_items = _get_unique_items()
    id_to_name = _get_id_to_name()
    
    def name_of(matrix_idx: int) -> Tuple[int, str]:
        """Return (bgg_id, name) for a matrix index"""
        if unique_items is None or id_to_name is None:
            return matrix_idx, f"<index_{matrix_idx}>"
        
        try:
            bgg_id = unique_items[matrix_idx]
            name = id_to_name.get(bgg_id, f"<id_{bgg_id}>")
            if pd.isna(name):
                name = str(bgg_id)
            return bgg_id, name
        except (IndexError, KeyError):
            return matrix_idx, f"<index_{matrix_idx}>"

    print(f"Top {max_print} most impacted games (metric disagreement on top-{use_k} neighbors):")
    for idx, (disagree, pos_diff_sum, i, inter, top_c, top_j) in enumerate(impacted[:max_print], 1):
        bgg_id, game_name = name_of(i)
        print(
            f"{idx}. BGG ID {bgg_id} ({game_name}): overlap={inter}, disagreement={disagree}, pos_diff_sum={pos_diff_sum}"
        )

    return {"best_game_id": best_i, "k": use_k}


In [41]:
# One-pass streaming preload: build reusable cache from CSV
import csv
import time
from heapq import heappush, heappushpop
from typing import Dict, List, Tuple, Optional, Iterable


class _BGGCache:
    def __init__(self,
                 path: str,
                 k: int,
                 delimiter: str,
                 header: List[str],
                 cols: Tuple[str, str, str, str, str],
                 topk_cosine: Dict[int, List[Tuple[float, int]]],
                 topk_jaccard: Dict[int, List[Tuple[float, int]]],
                 full_neighbors_by_i: Dict[int, Dict[int, Tuple[float, float]]],
                 stats: Dict[str, int]):
        self.path = path
        self.k = k
        self.delimiter = delimiter
        self.header = header
        self.cols = cols
        self.topk_cosine = topk_cosine
        self.topk_jaccard = topk_jaccard
        self.full_neighbors_by_i = full_neighbors_by_i
        self.stats = stats


_BGG_CACHE: Optional[_BGGCache] = None
# FIXED: Bidirectional preload that indexes edges in BOTH directions
import csv
import time
from heapq import heappush, heappushpop
from typing import Dict, List, Tuple, Optional, Iterable


def preload_streamed_index_bidirectional(
    path: str = '../data/bedges.csv',
    k: int = 10,
    focus_game_ids: Optional[Iterable[int]] = None,
    progress_every: int = 200_000,
    max_rows: Optional[int] = None,
):
    """Stream the CSV once to build reusable structures with BIDIRECTIONAL indexing.
    
    FIX: This version indexes each edge in BOTH directions (i→j AND j→i)
    to handle CSV files that may store edges in arbitrary direction.
    """
    global _BGG_CACHE

    if focus_game_ids is None:
        focus_set = set()
    else:
        focus_set = set(focus_game_ids)

    delim = _sniff_delimiter(path)
    print(
        f"Preloading BIDIRECTIONAL: {path} | delimiter='{delim}' | k={k} | focus_games={len(focus_set)}",
        flush=True,
    )

    t0 = time.time()
    row_count = 0
    used_rows = 0
    skipped_bad = 0

    topk_cosine: Dict[int, List[Tuple[float, int]]] = {}
    topk_jaccard: Dict[int, List[Tuple[float, int]]] = {}
    full_neighbors_by_i: Dict[int, Dict[int, Tuple[float, float]]] = {
        gi: {} for gi in focus_set
    }

    with open(path, newline='') as f:
        reader = csv.reader(f, delimiter=delim)
        try:
            raw_header = next(reader)
        except StopIteration:
            raise RuntimeError('Empty CSV file')
        header = _normalize_header(raw_header)
        colmap = _resolve_columns(header)
        if not colmap:
            raise RuntimeError(
                f"Unrecognized header: {raw_header} | normalized: {header}"
            )
        col_i, col_j, col_shared, col_cos, col_jac = colmap
        reader_dict = csv.DictReader(f, fieldnames=header, delimiter=delim)

        for row in reader_dict:
            row_count += 1
            if max_rows is not None and row_count > max_rows:
                break

            if row_count % progress_every == 0:
                elapsed = time.time() - t0
                rate = row_count / elapsed if elapsed > 0 else 0
                print(
                    f"Processed {row_count:,} | used={used_rows:,} | skipped={skipped_bad:,} | sources={len(topk_cosine):,} | {rate:,.0f} rows/s",
                    flush=True,
                )

            try:
                i = int(row[col_i])
                j = int(row[col_j])
                cos = float(row[col_cos])
                jac = float(row[col_jac])
            except Exception:
                skipped_bad += 1
                continue

            used_rows += 1

            # === FIX: Index in BOTH directions ===
            for src, dst in [(i, j), (j, i)]:
                # Maintain top-k heaps for cosine
                heap_c = topk_cosine.get(src)
                if heap_c is None:
                    heap_c = []
                    topk_cosine[src] = heap_c
                if len(heap_c) < k:
                    heappush(heap_c, (cos, dst))
                else:
                    if cos > heap_c[0][0]:
                        heappushpop(heap_c, (cos, dst))

                # Maintain top-k heaps for jaccard
                heap_j = topk_jaccard.get(src)
                if heap_j is None:
                    heap_j = []
                    topk_jaccard[src] = heap_j
                if len(heap_j) < k:
                    heappush(heap_j, (jac, dst))
                else:
                    if jac > heap_j[0][0]:
                        heappushpop(heap_j, (jac, dst))

                # Store full neighbors for focused games
                if src in full_neighbors_by_i:
                    prev = full_neighbors_by_i[src].get(dst)
                    if prev is None:
                        full_neighbors_by_i[src][dst] = (cos, jac)
                    else:
                        # Keep max of each metric
                        full_neighbors_by_i[src][dst] = (max(prev[0], cos), max(prev[1], jac))

    elapsed = time.time() - t0
    print(
        (
            f"Preload complete: rows={row_count:,} (used={used_rows:,}, skipped={skipped_bad:,}) in {elapsed:,.1f}s | "
            f"sources={len(topk_cosine):,} | focus_neighbors={{gi: len(ns) for gi, ns in full_neighbors_by_i.items()}}"
        ),
        flush=True,
    )

    _BGG_CACHE = _BGGCache(
        path=path,
        k=k,
        delimiter=delim,
        header=header,
        cols=(col_i, col_j, col_shared, col_cos, col_jac),
        topk_cosine=topk_cosine,
        topk_jaccard=topk_jaccard,
        full_neighbors_by_i=full_neighbors_by_i,
        stats={
            'rows': row_count,
            'used': used_rows,
            'skipped': skipped_bad,
            'sources': len(topk_cosine),
        },
    )

    return {
        'rows': row_count,
        'used': used_rows,
        'skipped': skipped_bad,
        'sources': len(topk_cosine),
        'k': k,
        'delimiter': delim,
        'focus_games': list(focus_set),
    }

In [42]:
# Preload ALL data (single pass). Uncomment the full run; subset is for a quick smoke test.
# Full run (can take a while on ~42M rows):
# preload_info = preload_streamed_index(
#     path='../data/bedges.csv',
#     k=10,
#     progress_every=1_000_000,
# )

# Smoke test on a subset; comment out after validating
preload_info = preload_streamed_index_bidirectional(
    path='../data/bedges.csv',
    k=100,
    progress_every=2_000_000,
    max_rows=None,
)
preload_info

Preloading BIDIRECTIONAL: ../data/bedges.csv | delimiter=';' | k=100 | focus_games=0
Processed 2,000,000 | used=1,999,999 | skipped=0 | sources=2,828 | 309,676 rows/s
Processed 4,000,000 | used=3,999,999 | skipped=0 | sources=4,264 | 311,355 rows/s
Processed 6,000,000 | used=5,999,999 | skipped=0 | sources=5,318 | 311,442 rows/s
Processed 8,000,000 | used=7,999,999 | skipped=0 | sources=6,204 | 308,486 rows/s
Processed 10,000,000 | used=9,999,999 | skipped=0 | sources=6,979 | 306,805 rows/s
Processed 12,000,000 | used=11,999,999 | skipped=0 | sources=7,767 | 305,855 rows/s
Processed 14,000,000 | used=13,999,999 | skipped=0 | sources=8,493 | 304,787 rows/s
Processed 16,000,000 | used=15,999,999 | skipped=0 | sources=9,176 | 302,718 rows/s
Processed 18,000,000 | used=17,999,999 | skipped=0 | sources=9,863 | 301,735 rows/s
Processed 20,000,000 | used=19,999,999 | skipped=0 | sources=10,482 | 301,059 rows/s
Processed 22,000,000 | used=21,999,999 | skipped=0 | sources=11,078 | 300,397 rows/

{'rows': 42502968,
 'used': 42502968,
 'skipped': 0,
 'sources': 16754,
 'k': 100,
 'delimiter': ';',
 'focus_games': []}

In [43]:
# Compute impact from cache (no re-read)
impact_summary = topk_impact_from_csv(k=100, max_print=20)
impact_summary

Computing impact for 16,754 sources with k=100 (preloaded k=100)…
Top 20 most impacted games (metric disagreement on top-100 neighbors):
1. BGG ID 152959.0 (The Settlers of Catan): overlap=5, disagreement=190, pos_diff_sum=197
2. BGG ID 142057.0 (Carcassonne Big Box): overlap=5, disagreement=190, pos_diff_sum=144
3. BGG ID 140709.0 (Alhambra: Family Box): overlap=8, disagreement=184, pos_diff_sum=216
4. BGG ID 38821.0 (Settlers of Catan: Gallery Edition): overlap=9, disagreement=182, pos_diff_sum=403
5. BGG ID 201856.0 (Last Will): overlap=9, disagreement=182, pos_diff_sum=337
6. BGG ID 315048.0 (Survive: Escape from Atlantis!): overlap=10, disagreement=180, pos_diff_sum=497
7. BGG ID 175961.0 (Three Cheers for Master): overlap=10, disagreement=180, pos_diff_sum=291
8. BGG ID 147240.0 (Catan: Family Edition): overlap=10, disagreement=180, pos_diff_sum=290
9. BGG ID 152241.0 (Ultimate Werewolf): overlap=10, disagreement=180, pos_diff_sum=168
10. BGG ID 128666.0 (BANG! 10th Anniversary):

{'best_game_id': 7789, 'k': 100}

In [44]:
# Investigate metric differences for a single game (default: 7789) using preloaded cache
import math
import csv
import pandas as pd
import numpy as np
from typing import Optional, Dict, Tuple

# Cached lookups for matrix index ↔ bgg_id and bgg_id → name
_UNIQUE_ITEMS = None
_ID_TO_NAME = None


def _get_unique_items():
    """Get or reconstruct unique_items mapping (matrix_index → bgg_id)"""
    global _UNIQUE_ITEMS
    
    # First, try to use global unique_items if available
    if 'unique_items' in globals():
        return globals()['unique_items']
    
    # If cached, return it
    if _UNIQUE_ITEMS is not None:
        return _UNIQUE_ITEMS
    
    # Otherwise, reconstruct it from the rating data
    try:
        df = pd.read_csv('../data/bgg_rating_threshold.csv', sep=';', usecols=['bgg_id'])
        _, unique_items = pd.factorize(df['bgg_id'], sort=True)
        _UNIQUE_ITEMS = unique_items
        return unique_items
    except Exception as e:
        print(f"Warning: Could not reconstruct unique_items: {e}")
        return None


def _get_id_to_name():
    """Get name mapping (bgg_id → game_name)"""
    global _ID_TO_NAME
    
    # First, try to use global id_to_name if available
    if 'id_to_name' in globals():
        return globals()['id_to_name']
    
    # If cached, return it
    if _ID_TO_NAME is not None:
        return _ID_TO_NAME
    
    # Otherwise, load from boardgames_ranks.csv
    try:
        df_game_item_from_bgg = pd.read_csv('../data/boardgames_ranks.csv')
        id_to_name = df_game_item_from_bgg.set_index('id')['name']
        _ID_TO_NAME = id_to_name
        return id_to_name
    except Exception as e:
        print(f"Warning: Could not load id_to_name: {e}")
        return None


def _bgg_id_to_matrix_idx(bgg_id: int) -> Optional[int]:
    """Convert BGG ID to matrix index"""
    unique_items = _get_unique_items()
    if unique_items is None:
        return None
    
    try:
        # Find the index where bgg_id appears in unique_items
        idx = np.where(unique_items == bgg_id)[0]
        if len(idx) > 0:
            return int(idx[0])
        return None
    except Exception as e:
        print(f"Warning: Could not convert bgg_id {bgg_id} to matrix index: {e}")
        return None


def inspect_game_metrics(
    bgg_id: int = 13,
    k: int = 20,
):
    """Inspect metric differences for a single game.
    
    Args:
        bgg_id: BoardGameGeek ID (e.g., 13 for Catan)
        k: Number of top neighbors to compare
    """
    if '_BGG_CACHE' not in globals() or _BGG_CACHE is None:
        raise RuntimeError("Cache not found. Run `preload_streamed_index(...)` first.")
    cache = _BGG_CACHE

    # Convert BGG ID to matrix index
    game_id = _bgg_id_to_matrix_idx(bgg_id)
    if game_id is None:
        raise ValueError(f"BGG ID {bgg_id} not found in the dataset")

    # If we have full neighbors for this game, use them; else fall back to union of top-k lists
    neighbors: Optional[Dict[int, Tuple[float, float]]] = cache.full_neighbors_by_i.get(game_id)
    if neighbors:
        all_items = list(neighbors.items())  # (neighbor, (cos, jac))
        sorted_cos = sorted(all_items, key=lambda x: x[1][0], reverse=True)
        sorted_jac = sorted(all_items, key=lambda x: x[1][1], reverse=True)
    else:
        # Fallback: use the preloaded top-k heaps for this game only
        heap_c = cache.topk_cosine.get(game_id, [])
        heap_j = cache.topk_jaccard.get(game_id, [])
        sorted_cos = sorted(heap_c, key=lambda x: x[0], reverse=True)
        sorted_jac = sorted(heap_j, key=lambda x: x[0], reverse=True)
        # Convert to uniform (nid, (cos, jac)) tuples; unknown metric becomes None
        def to_pairs(sorted_list, which):
            out = []
            for score, nid in sorted_list:
                if which == 'cos':
                    out.append((nid, (score, None)))
                else:
                    out.append((nid, (None, score)))
            return out
        # Merge by nid keeping max of each metric
        tmp: Dict[int, Tuple[Optional[float], Optional[float]]] = {}
        for nid, (c, j) in to_pairs(sorted_cos, 'cos') + to_pairs(sorted_jac, 'jac'):
            prev = tmp.get(nid)
            if prev is None:
                tmp[nid] = (c, j)
            else:
                c_prev, j_prev = prev
                tmp[nid] = (max(c_prev or float('-inf'), c or float('-inf')) if (c_prev is not None or c is not None) else None,
                            max(j_prev or float('-inf'), j or float('-inf')) if (j_prev is not None or j is not None) else None)
        # Fill Nones with -inf to safely sort
        all_items = [(nid, (c if c is not None else float('-inf'), j if j is not None else float('-inf'))) for nid, (c, j) in tmp.items()]
        sorted_cos = sorted(all_items, key=lambda x: x[1][0], reverse=True)
        sorted_jac = sorted(all_items, key=lambda x: x[1][1], reverse=True)

    rankC = {nid: r for r, (nid, _) in enumerate(sorted_cos, 1)}
    rankJ = {nid: r for r, (nid, _) in enumerate(sorted_jac, 1)}

    topC = [nid for nid, _ in sorted_cos[:k]]
    topJ = [nid for nid, _ in sorted_jac[:k]]

    setC, setJ = set(topC), set(topJ)
    inter = setC & setJ
    disagree = len(setC) + len(setJ) - 2 * len(inter)

    # Spearman on overlap ranks
    rho = None
    if inter:
        x = [rankC[nid] for nid in inter]
        y = [rankJ[nid] for nid in inter]
        n = len(x)
        if n > 1:
            meanx = sum(x) / n
            meany = sum(y) / n
            num = sum((a - meanx) * (b - meany) for a, b in zip(x, y))
            denx = math.sqrt(sum((a - meanx) ** 2 for a in x))
            deny = math.sqrt(sum((b - meany) ** 2 for b in y))
            rho = num / (denx * deny) if denx and deny else 0.0
        else:
            rho = 1.0

    # Human-readable printing with names using the same mapping as the graph
    unique_items = _get_unique_items()
    id_to_name = _get_id_to_name()
    
    def name_of(matrix_idx: int) -> Tuple[int, str]:
        """Return (bgg_id, name) for a matrix index"""
        if unique_items is None or id_to_name is None:
            return matrix_idx, f"<index_{matrix_idx}>"
        
        try:
            bgg_id = unique_items[matrix_idx]
            name = id_to_name.get(bgg_id, f"<id_{bgg_id}>")
            if pd.isna(name):
                name = str(bgg_id)
            return bgg_id, name
        except (IndexError, KeyError):
            return matrix_idx, f"<index_{matrix_idx}>"
    
    root_bgg_id, root_name = name_of(game_id)

    print(
        f"Summary for matrix_index={game_id} | bgg_id={root_bgg_id} | {root_name} | neighbors={len(sorted_cos)} | overlap(top-{k})={len(inter)} | disagreement={disagree} | spearman_rho_on_overlap={rho}",
        flush=True,
    )

    print(f"\nTop-{k} by cosine vs jaccard (union):")
    # Print formatted table header
    print(f"{'matrix_idx':<12} {'bgg_id':<8} {'name':<50} {'cos_rank':<10} {'jac_rank':<10} {'delta':<10}")
    print("-" * 110)
    
    union = list(setC | setJ)
    union.sort(key=lambda nid: min(rankC.get(nid, 10**9), rankJ.get(nid, 10**9)))
    for nid in union:
        rc = rankC.get(nid)
        rj = rankJ.get(nid)
        dr = (rc - rj) if (rc is not None and rj is not None) else '-'
        nbgg, nname = name_of(nid)
        # Truncate long names
        nname_display = nname[:47] + '...' if len(nname) > 50 else nname
        rc_display = str(rc) if rc is not None else '-'
        rj_display = str(rj) if rj is not None else '-'
        print(f"{nid:<12} {nbgg:<8} {nname_display:<50} {rc_display:<10} {rj_display:<10} {dr:<10}")

    return 


# Example: inspect using the cache (requires preloaded focus for full neighbors)
# BGG ID 13 is The Settlers of Catan
inspect_game_metrics(bgg_id=29352, k=100)


Summary for matrix_index=4476 | bgg_id=29352.0 | Lupusburg | neighbors=136 | overlap(top-100)=64 | disagreement=72 | spearman_rho_on_overlap=0.668537419917285

Top-100 by cosine vs jaccard (union):
matrix_idx   bgg_id   name                                               cos_rank   jac_rank   delta     
--------------------------------------------------------------------------------------------------------------
5582         63539.0  Lupus in Tabula                                    1          1          0         
5061         39219.0  Turandot                                           2          2          0         
3023         9801.0   Ostrakon                                           3          3          0         
4281         25951.0  The Castle of the Devil                            4          33         -29       
5520         59335.0  Wherewolf                                          6          4          2         
5383         50862.0  Caligula                         